In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

To use the fer2013 dataset, we change directory to where it is located. I use my google drive for simplicity when taking advantage of Google colab's GPU's

In [2]:
%cd /content/drive/My Drive/Colab Notebooks/emotion_detector_test/data/

/content/drive/My Drive/Colab Notebooks/emotion_detector_test/data


First, we define a pytorch dataset class for the FER2013 dataset. Training images are resized to 48x48 and we introduce random horizontal flipping, rotation and cropping to reduce risk of overfitting.

In [3]:
class FER2013Dataset(Dataset):
    def __init__(self, csv_file, usage = "Training", transform = None):
        self.data = pd.read_csv(csv_file)
        self.data = self.data[self.data['Usage'] == usage]
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        emotion = int(self.data.iloc[idx]['emotion'])
        pixels = self.data.iloc[idx]['pixels']
        img = np.array(pixels.split(), dtype="uint8").reshape(48,48)

        #Convert to torch tensor and apply transform if needed
        if self.transform:
            img = self.transform(img)
        else:
            img = torch.tensor(img, dtype=torch.float32).unsqueeze(0) / 255.0

        return img, emotion

# Image transformations

train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((48,48)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(5),
    transforms.RandomCrop((48,48), padding=1),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

test_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((48,48)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])


Next we define test and train sets and load the data:

In [4]:
train_data = FER2013Dataset(csv_file="fer2013.csv", usage="Training", transform=train_transform)
test_data  = FER2013Dataset(csv_file="fer2013.csv", usage="PublicTest", transform=test_transform)

###LOAD DATA###
train_loader = torch.utils.data.DataLoader(train_data, batch_size = 64, shuffle = True)
test_loader = torch.utils.data.DataLoader(test_data, batch_size = 64, shuffle = False)

We build a simple 3 layer Convolutional neural network to use as our classifier model:

In [5]:
class EmotionCNN(nn.Module):
    def __init__(self, num_classes = 7):
        super(EmotionCNN, self).__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(in_channels = 1, out_channels = 32, kernel_size = 3, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(in_channels = 32, out_channels = 64, kernel_size = 3, padding = 1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(in_channels = 64, out_channels = 128, kernel_size = 3, padding = 1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.fc_layers = nn.Sequential(
            nn.Linear(128*6*6, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )
    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layers(x)
        return x

Next, we prepare and design the training loop. I arbitrarily use 50 epochs and a learning rate of 0.001

In [6]:
# use GPU for computations if possible
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = EmotionCNN().to(device)

###DEFINE LOSS AND OPTIMISER###
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = 0.001)

###TRAINING LOOP###
num_epochs = 50
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch [{epoch + 1}/{num_epochs}], Loss: {running_loss / len(train_loader):.4f}")


Epoch [1/50], Loss: 1.7682
Epoch [2/50], Loss: 1.5823
Epoch [3/50], Loss: 1.5048
Epoch [4/50], Loss: 1.4626
Epoch [5/50], Loss: 1.4360
Epoch [6/50], Loss: 1.4072
Epoch [7/50], Loss: 1.3877
Epoch [8/50], Loss: 1.3704
Epoch [9/50], Loss: 1.3539
Epoch [10/50], Loss: 1.3465
Epoch [11/50], Loss: 1.3268
Epoch [12/50], Loss: 1.3233
Epoch [13/50], Loss: 1.3135
Epoch [14/50], Loss: 1.3108
Epoch [15/50], Loss: 1.2986
Epoch [16/50], Loss: 1.2826
Epoch [17/50], Loss: 1.2822
Epoch [18/50], Loss: 1.2737
Epoch [19/50], Loss: 1.2632
Epoch [20/50], Loss: 1.2615
Epoch [21/50], Loss: 1.2459
Epoch [22/50], Loss: 1.2421
Epoch [23/50], Loss: 1.2401
Epoch [24/50], Loss: 1.2377
Epoch [25/50], Loss: 1.2266
Epoch [26/50], Loss: 1.2232
Epoch [27/50], Loss: 1.2093
Epoch [28/50], Loss: 1.2057
Epoch [29/50], Loss: 1.2033
Epoch [30/50], Loss: 1.1922
Epoch [31/50], Loss: 1.1872
Epoch [32/50], Loss: 1.1849
Epoch [33/50], Loss: 1.1709
Epoch [34/50], Loss: 1.1806
Epoch [35/50], Loss: 1.1741
Epoch [36/50], Loss: 1.1693
E

Time for model evaluation:

Here, we run an experiment to evaluate model weight quantization settings on model accuracy with settings (per channel and per tensor) full FP32, 8 bits, 6 bits, 4 bits and 2 bits.

In [8]:

import copy

# Evaluate any version of the trained model
def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0

    with torch.inference_mode():
        for images, labels in loader:
            outputs = model(images.to("cpu"))
            predictions = outputs.argmax(dim=1)

            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    return 100 * correct / total


# Simulate symmetric weight quantization
def quantize_weights(model, bits, per_channel=False):
    quant_model = copy.deepcopy(model).cpu().eval()

    qmax = 2 ** (bits - 1) - 1

    with torch.no_grad():
        for layer in quant_model.modules():
            if isinstance(layer, (nn.Conv2d, nn.Linear)):
                weights = layer.weight

                if per_channel:
                    # Separate scale for each output channel
                    dims = tuple(range(1, weights.ndim))
                    max_abs = weights.abs().amax(
                        dim=dims, keepdim=True
                    )
                else:
                    # One scale for the entire weight tensor
                    max_abs = weights.abs().max()

                scale = (max_abs / qmax).clamp(min=1e-12)

                quantized = torch.round(weights / scale)
                quantized = torch.clamp(
                    quantized, -qmax, qmax
                )

                # Dequantize back to floating point
                weights.copy_(quantized * scale)

    return quant_model


# Run experiments
results = []

baseline = evaluate(model.cpu(), test_loader)

results.append({
    "Bits": 32,
    "Granularity": "FP32 baseline",
    "Accuracy": baseline
})

for bits in [8, 6, 4, 2]:
    for granularity in ["per_tensor", "per_channel"]:

        quant_model = quantize_weights(
            model,
            bits=bits,
            per_channel=(granularity == "per_channel")
        )

        accuracy = evaluate(quant_model, test_loader)

        results.append({
            "Bits": bits,
            "Granularity": granularity,
            "Accuracy": accuracy
        })

df = pd.DataFrame(results)
df["Accuracy drop (pp)"] = baseline - df["Accuracy"]

print(df.to_string(index=False))


 Bits   Granularity  Accuracy  Accuracy drop (pp)
   32 FP32 baseline 60.267484            0.000000
    8    per_tensor 59.933129            0.334355
    8   per_channel 60.211758            0.055726
    6    per_tensor 60.128169            0.139315
    6   per_channel 59.738089            0.529395
    4    per_tensor 57.063249            3.204235
    4   per_channel 57.954862            2.312622
    2    per_tensor 14.098635           46.168849
    2   per_channel 19.866258           40.401226
